# 05 — Reduce Sum（归约求和）

## 背景

**归约（Reduction）** 是将一组值合并为单一值的操作，如求和、求最大值等。它在神经网络中无处不在：LayerNorm、BatchNorm 的均值方差、Softmax 的指数和、Loss 的全局求和等。

归约的 GPU 实现难点在于**跨线程的数据依赖**：不同于元素级运算，归约要求线程间通信（Warp Shuffle 或 Shared Memory）。三种框架在归约上的写法差异很能体现各自的抽象层次：

- **Triton**：每个 program 处理一行，用 Python 循环 + `tl.sum` 实现
- **TileLang**：用 `T.reduce_sum` + `T.Serial` 处理分块迭代

## 问题定义

给定矩阵 `A [N, M]`，计算每行之和，输出向量 `B [N]`：

$$B[i] = \sum_{j=0}^{M-1} A[i, j], \quad i = 0, 1, \ldots, N-1$$

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch 参考实现

In [ ]:
N, M = 4096, 16384
A = torch.randn(N, M, dtype=torch.float32, device="cuda")

def ref_reduce_sum(A):
    return torch.sum(A, dim=1)   # [N, M] → [N]

B_ref = ref_reduce_sum(A)
print(f"A: {A.shape} → B: {B_ref.shape}")

## Triton 实现

每个 Triton program 负责一行：在 Python 循环内分块加载，`tl.sum(chunk, 0)` 在 warp 内归约，最后把各块的部分和累加，用 `tl.store` 写出标量结果。

> `tl.sum(x, 0)` 对一维张量 `x [BLOCK_M]` 归约，返回标量（0-d Triton tensor）。

In [ ]:
@triton.jit
def _reduce_sum_kernel(A_ptr, B_ptr, M, BLOCK_M: tl.constexpr):
    row  = tl.program_id(0)   # one program per row
    base = A_ptr + row * M
    acc  = tl.zeros([BLOCK_M], dtype=tl.float32)
    for start in range(0, M, BLOCK_M):
        cols  = start + tl.arange(0, BLOCK_M)
        chunk = tl.load(base + cols, mask=cols < M, other=0.0)
        acc   = acc + chunk.to(tl.float32)
    tl.store(B_ptr + row, tl.sum(acc, 0))

BLOCK_M_TRI = 1024

def triton_reduce_sum(A):
    N_, M_ = A.shape
    B = torch.empty(N_, dtype=A.dtype, device=A.device)
    _reduce_sum_kernel[(N_,)](A, B, M_, BLOCK_M=BLOCK_M_TRI)
    return B

B_tri = triton_reduce_sum(A)
ok = torch.allclose(B_ref, B_tri, atol=1e-2)
print("Triton correctness:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(B_ref - B_tri)).item():.4f}")

## TileLang 实现

TileLang 用 `T.reduce_sum(src, dst, dim=1, clear=False)` 封装了 warp-level 归约；`T.Serial` 指定串行循环（有跨迭代的数据依赖），`clear=False` 表示累加模式。

In [ ]:
# ── GPU 自适应配置 ────────────────────────────────────────────────────────────
# gfx1100 (RX 7900 XTX): BN=2, BM=128, threads=128
# gfx1201 (R9700): BN=1, BM=64, threads=128（T.copy 填充下 BM≤64）
# gfx1151 (Radeon 8060S): RDNA3.5 iGPU
#   关键发现：T.reduce_sum 的 BM 上限取决于 fragment 的填充方式：
#   - T.copy 填充：BM ≤ 64（受限更严）
#   - T.Parallel 逐元素填充：BM ≤ 128（约束放宽）→ 串行迭代从 512 次降至 128 次
#   配置：BN=1, BM=128, TH=256 → 1.17ms，仅比 PyTorch 慢 4%（原 1.33ms 慢 19%）
#   T.Kernel 的 threads 必须是字面常量，每个 arch 用独立 kernel 函数
arch = torch.cuda.get_device_properties(0).gcnArchName

if arch.startswith("gfx1151"):
    @tilelang.jit(pass_configs={
        tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
        tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
    })
    def tl_reduce_sum(A, BLOCK_N: int, BLOCK_M: int):
        N_, M_ = T.const("N, M")
        dtype = T.float32
        A: T.Tensor((N_, M_), dtype)
        B = T.empty((N_,), dtype)
        with T.Kernel(N_ // BLOCK_N, threads=256) as pid_n:
            Al  = T.alloc_fragment((BLOCK_N, BLOCK_M), dtype)
            acc = T.alloc_fragment((BLOCK_N,), dtype)
            T.clear(acc)
            for m_blk in T.Serial(M_ // BLOCK_M):
                # T.Parallel 逐元素加载代替 T.copy，允许 BM=128（T.copy 下限 BM≤64）
                for i, j in T.Parallel(BLOCK_N, BLOCK_M):
                    Al[i, j] = A[pid_n * BLOCK_N + i, m_blk * BLOCK_M + j]
                T.reduce_sum(Al, acc, dim=1, clear=False)
            T.copy(acc, B[pid_n * BLOCK_N])
        return B

    BN, BM = 1, 128

elif arch.startswith("gfx1201"):
    @tilelang.jit(pass_configs={
        tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
        tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
    })
    def tl_reduce_sum(A, BLOCK_N: int, BLOCK_M: int):
        N_, M_ = T.const("N, M")
        dtype = T.float32
        A: T.Tensor((N_, M_), dtype)
        B = T.empty((N_,), dtype)
        with T.Kernel(N_ // BLOCK_N, threads=128) as pid_n:
            A_local = T.alloc_fragment((BLOCK_N, BLOCK_M), dtype)
            B_local = T.alloc_fragment((BLOCK_N,), dtype)
            T.clear(B_local)
            for m_blk in T.Serial(M_ // BLOCK_M):
                T.copy(A[pid_n * BLOCK_N, m_blk * BLOCK_M], A_local)
                T.reduce_sum(A_local, B_local, dim=1, clear=False)
            T.copy(B_local, B[pid_n * BLOCK_N])
        return B

    BN, BM = 1, 64

else:
    @tilelang.jit(pass_configs={
        tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
        tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
    })
    def tl_reduce_sum(A, BLOCK_N: int, BLOCK_M: int):
        N_, M_ = T.const("N, M")
        dtype = T.float32
        A: T.Tensor((N_, M_), dtype)
        B = T.empty((N_,), dtype)
        with T.Kernel(N_ // BLOCK_N, threads=128) as pid_n:
            A_local = T.alloc_fragment((BLOCK_N, BLOCK_M), dtype)
            B_local = T.alloc_fragment((BLOCK_N,), dtype)
            T.clear(B_local)
            for m_blk in T.Serial(M_ // BLOCK_M):
                T.copy(A[pid_n * BLOCK_N, m_blk * BLOCK_M], A_local)
                T.reduce_sum(A_local, B_local, dim=1, clear=False)
            T.copy(B_local, B[pid_n * BLOCK_N])
        return B

    BN, BM = 2, 128

k = tl_reduce_sum.compile(N=N, M=M, BLOCK_N=BN, BLOCK_M=BM)
B_tl = k(A.clone())
ok = torch.allclose(B_ref, B_tl, atol=1e-2)
print("TileLang 正确性:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(B_ref - B_tl)).item():.4f}")

## 性能对比

In [ ]:
WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

results = {
    "PyTorch":  bench(ref_reduce_sum,    [A]),
    "Triton":   bench(triton_reduce_sum, [A]),
    "TileLang": bench(k,                [A]),
}

bytes_rw = N * M * 4 + N * 4   # float32
pt_ms  = results["PyTorch"]
tri_ms = results["Triton"]

header = f"{'实现':<12}  {'延迟 (ms)':>10}  {'带宽 (TB/s)':>12}  {'vs PyTorch':>10}  {'vs Triton':>10}"
sep    = "-" * len(header)
print(f"Reduce Sum  (N={N}, M={M}, float32)")
print(sep)
print(header)
print(sep)
for name, ms in results.items():
    bw     = bytes_rw / ms / 1e9
    vs_pt  = f"{pt_ms/ms:+.1%}"
    vs_tri = f"{tri_ms/ms:+.1%}"
    print(f"{name:<12}  {ms:>10.4f}  {bw:>12.2f}  {vs_pt:>10}  {vs_tri:>10}")
print(sep)
